# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** de vos variables quantitatives et justifier l'emploi d'indicateurs robustes (**Médiane `PERCENTILE_APPROX`**, **IQR `P75 - P25`**, **P90**) plutôt que la seule moyenne arithmétique.
2. Écrire des requêtes **Spark SQL avancées** mobilisant des CTE (`WITH ...`) et des **fonctions de fenêtrage** (`DENSE_RANK`, `ROW_NUMBER`, cumul de **Pareto**, variation temporelle **`LAG`**).
3. Construire au moins **2 à 3 tables agrégées métier dans `/lakehouse/gold/`** prêtes à alimenter votre Dashboard Streamlit à faible latence.
4. Éviter le *Small Files Problem* sur HDFS et compacter votre table Silver avec `OPTIMIZE`.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("J3_Analytics_SQL_Gold")
    .getOrCreate()
)

# Table Silver principale exposée en vue SQL `v_silver`
SILVER_MAIN_PATH = "/lakehouse/silver/rides"  # courses propres enrichies (météo + gamme), cf. Jour 2
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
print("Lignes disponibles dans v_silver :", spark.sql("SELECT COUNT(*) FROM v_silver").collect()[0][0])

Lignes disponibles dans v_silver : 637977


## 1. Analyse d'Asymétrie : Moyenne vs Médiane & Écart Interquartile (IQR)

### Exercice 1.1 — Étudier la distribution de votre métrique quantitative principale
Écrivez une requête Spark SQL sur `v_silver` calculant sur votre variable continue principale :
- `COUNT(*)`, `AVG(...)`, `STDDEV(...)`, `SKEWNESS(...)`
- Les quantiles `P25`, `P50` (médiane), `P75`, `P90`, `P95` via `PERCENTILE_APPROX(col, q)` et `MAX(...)`.

Rédigez ensuite en Markdown **votre interprétation statistique** (écart moyenne/médiane, impact des valeurs extrêmes, choix de l'indicateur retenu pour votre dashboard).

In [2]:
# Résumé statistique en Spark SQL : prix (métrique principale), comparé au prix au mile et à la distance
GOLD_RESUME_PATH = "/lakehouse/gold/resume_distributions"
df_dist_summary = spark.sql("""
    WITH mesures AS (
        SELECT 'price' AS variable, price AS valeur FROM v_silver
        UNION ALL SELECT 'prix_par_mile', prix_par_mile FROM v_silver
        UNION ALL SELECT 'distance', distance FROM v_silver
    )
    SELECT variable,
           COUNT(*)                                AS nb,
           ROUND(AVG(valeur), 2)                   AS moyenne,
           ROUND(STDDEV(valeur), 2)                AS ecart_type,
           ROUND(SKEWNESS(valeur), 3)              AS skewness,
           PERCENTILE_APPROX(valeur, 0.25)         AS p25,
           PERCENTILE_APPROX(valeur, 0.50)         AS p50_mediane,
           PERCENTILE_APPROX(valeur, 0.75)         AS p75,
           PERCENTILE_APPROX(valeur, 0.90)         AS p90,
           PERCENTILE_APPROX(valeur, 0.95)         AS p95,
           MAX(valeur)                             AS max,
           ROUND(PERCENTILE_APPROX(valeur, 0.75) - PERCENTILE_APPROX(valeur, 0.25), 2) AS iqr,
           ROUND(AVG(valeur) - PERCENTILE_APPROX(valeur, 0.50), 2)                   AS ecart_moyenne_mediane
    FROM mesures
    GROUP BY variable
    ORDER BY variable DESC
""")
df_dist_summary.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_RESUME_PATH)
assert df_dist_summary is not None, "TODO : Écrivez la requête SQL de résumé statistique."
df_dist_summary.show(truncate=False)

# Asymétrie selon la compagnie (le prix est-il asymétrique partout ?)
spark.sql("""
    SELECT cab_type, COUNT(*) AS nb, ROUND(AVG(price), 2) AS moyenne, PERCENTILE_APPROX(price, 0.5) AS mediane,
           ROUND(SKEWNESS(price), 3) AS skewness
    FROM v_silver GROUP BY cab_type ORDER BY cab_type
""").show()

+-------------+------+-------+----------+--------+----+-----------+-----+-----+-----+------+----+---------------------+
|variable     |nb    |moyenne|ecart_type|skewness|p25 |p50_mediane|p75  |p90  |p95  |max   |iqr |ecart_moyenne_mediane|
+-------------+------+-------+----------+--------+----+-----------+-----+-----+-----+------+----+---------------------+
|prix_par_mile|637977|9.69   |13.71     |40.743  |4.66|7.49       |11.54|17.24|23.31|1375.0|6.88|2.2                  |
|price        |637977|16.55  |9.32      |1.046   |9.0 |13.5       |22.5 |30.0 |34.0 |97.5  |13.5|3.05                 |
|distance     |637977|2.19   |1.14      |0.817   |1.27|2.16       |2.93 |3.41 |4.45 |7.86  |1.66|0.03                 |
+-------------+------+-------+----------+--------+----+-----------+-----+-----+-----+------+----+---------------------+



+--------+------+-------+-------+--------+
|cab_type|    nb|moyenne|mediane|skewness|
+--------+------+-------+-------+--------+
|    Lyft|307409|  17.35|   16.5|   0.978|
|    Uber|330568|   15.8|   12.5|   1.053|
+--------+------+-------+-------+--------+



**Votre justification statistique *(À compléter)* :**
> - **Prix** : skewness = **1,05**, asymétrie à droite nette. La moyenne (16,55 $) dépasse la médiane (13,50 $) de **3,05 $**, car une minorité de courses premium longues (P95 = 34 $, max = 97,50 $) tire la moyenne vers le haut. La moitié des courses coûtent entre 9 $ et 22,50 $ (IQR = 13,50 $).
> - **Prix au mile** : skewness = **40,7**, asymétrie extrême. Les trajets de quelques centaines de mètres ont un prix fixe minimum, ce qui donne jusqu'à 1 375 $ « au mile ». La moyenne (9,69 $) n'a ici aucun sens, la médiane (7,49 $) si.
> - **Distance** : skewness = 0,82, moyenne ≈ médiane (2,19 vs 2,16 miles) : la distance est peu asymétrique.
> - Les deux compagnies ont la même forme de distribution (skewness 0,98 chez Lyft, 1,05 chez Uber), mais l'écart moyenne / médiane est bien plus fort chez Uber (15,80 $ vs 12,50 $).
>
> **Choix pour le dashboard :** **médiane** + **IQR** (et P90 pour décrire le haut de gamme), plutôt que moyenne et écart-type, trop sensibles aux extrêmes.

---
## 2. Table Gold #1 : Classement Métier & Loi de Pareto (`OVER ORDER BY ...`)

### Exercice 2.1 — Agrégation par segment/catégorie + Courbe cumulée de Pareto
Construisez en Spark SQL une première table Gold agrégée par une dimension métier pertinente de votre dataset incluant :
- Le volume d'observations, le total et les indicateurs robustes (**médiane**, **IQR**).
- Le rang via `DENSE_RANK() OVER (ORDER BY ... DESC)`.
- Le pourcentage cumulé du total (**analyse de Pareto 80/20**) via :
  `SUM(metric_total) OVER (ORDER BY metric_total DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(metric_total) OVER ()`.

Sauvegardez-la avec `.coalesce(1)` au format **Delta** dans `/lakehouse/gold/<nom_gold_1>`.

In [3]:
GOLD_PARETO_PATH = "/lakehouse/gold/pareto_trajets"  # 1 ligne = 1 trajet (quartier de départ -> d'arrivée)

df_gold_1 = spark.sql("""
    WITH par_trajet AS (
        SELECT trajet, source, destination,
               COUNT(*)                                                         AS nb_courses,
               ROUND(SUM(price), 2)                                             AS ca_total,
               PERCENTILE_APPROX(price, 0.5)                                    AS prix_median,
               PERCENTILE_APPROX(price, 0.75) - PERCENTILE_APPROX(price, 0.25)  AS iqr_prix,
               PERCENTILE_APPROX(price, 0.9)                                    AS p90_prix,
               PERCENTILE_APPROX(distance, 0.5)                                 AS distance_mediane,
               PERCENTILE_APPROX(prix_par_mile, 0.5)                            AS prix_mile_median,
               ROUND(100 * AVG(is_surge), 2)                                    AS pct_majorees
        FROM v_silver
        GROUP BY trajet, source, destination
    )
    SELECT *,
           DENSE_RANK() OVER (ORDER BY ca_total DESC)                           AS rang_ca,
           ROUND(100 * ca_total / SUM(ca_total) OVER (), 2)                     AS part_ca_pct,
           ROUND(100 * SUM(ca_total) OVER (ORDER BY ca_total DESC
                                           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
                 / SUM(ca_total) OVER (), 2)                                    AS pareto_cumul_pct,
           ROUND(100 * ROW_NUMBER() OVER (ORDER BY ca_total DESC) / COUNT(*) OVER (), 2) AS pct_trajets_cumul
    FROM par_trajet
    ORDER BY rang_ca
""")
assert df_gold_1 is not None, "TODO : Écrivez la requête SQL avec fenêtre de Pareto dans `df_gold_1`."

# Sauvegarde Delta en 1 seul fichier (petite table, lue par le dashboard)
df_gold_1.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_PARETO_PATH)
df_gold_1.show(15, truncate=False)

# Lecture Pareto : quelle part des trajets fait 80 % du chiffre d'affaires simulé ?
seuil = df_gold_1.filter("pareto_cumul_pct >= 80").orderBy("rang_ca").first()
nb_trajets = df_gold_1.count()
print(f"{nb_trajets} trajets | 80 % du CA atteint au trajet n°{seuil['rang_ca']} "
      f"soit {seuil['pct_trajets_cumul']} % des trajets (une loi 80/20 stricte donnerait 20 %)")

+--------------------------------------------+-----------------------+-----------------------+----------+---------+-----------+--------+--------+----------------+----------------+------------+-------+-----------+----------------+-----------------+
|trajet                                      |source                 |destination            |nb_courses|ca_total |prix_median|iqr_prix|p90_prix|distance_mediane|prix_mile_median|pct_majorees|rang_ca|part_ca_pct|pareto_cumul_pct|pct_trajets_cumul|
+--------------------------------------------+-----------------------+-----------------------+----------+---------+-----------+--------+--------+----------------+----------------+------------+-------+-----------+----------------+-----------------+
|Financial District → Boston University      |Financial District     |Boston University      |8940      |227956.0 |24.0       |22.5    |45.5    |5.37            |4.68            |3.02        |1      |2.16       |2.16            |1.39             |
|Boston 

72 trajets | 80 % du CA atteint au trajet n°54 soit 75.0 % des trajets (une loi 80/20 stricte donnerait 20 %)


**Lecture de la table Gold #1 (`/lakehouse/gold/pareto_trajets`)**
- Les trajets qui rapportent le plus sont les **plus longs** : Financial District ↔ Boston University (5,4 miles, prix médian 24 $), Fenway ↔ Financial District… Le classement par CA est d'abord un classement par **distance**.
- **Pas de loi de Pareto :** il faut **54 trajets sur 72 (75 %)** pour atteindre 80 % du CA, loin des 20 % d'une loi 80/20. Chaque trajet a presque le même nombre de courses (de 8 418 à 9 534) : les données viennent de requêtes **régulières** aux API sur tous les trajets, pas de vraies réservations. La concentration du CA ne vient donc que des écarts de prix entre trajets.
- Les indicateurs robustes par trajet (médiane, IQR, P90) montrent que l'IQR d'un trajet (de 8,50 $ pour les plus courts à 22,50 $ pour les plus longs) vient surtout du **choix de la gamme** (Shared ou Black SUV sur le même trajet).

## 3. Table Gold #2 : Dynamique Temporelle & Croissance avec `LAG()`

### Exercice 3.1 — Série temporelle et taux d'évolution période-sur-période
Agrégez votre table `v_silver` selon son axe temporel (jour, semaine, mois ou année selon votre sujet) et utilisez la fonction de fenêtrage **`LAG(metric, 1) OVER (ORDER BY periode)`** (ou une moyenne glissante `AVG(...) OVER (ORDER BY periode ROWS BETWEEN ... PRECEDING AND CURRENT ROW)`) pour mesurer la dynamique d'évolution.

Sauvegardez le résultat dans `/lakehouse/gold/<nom_gold_2>`.

In [4]:
GOLD_TIME_PATH = "/lakehouse/gold/evolution_journaliere"  # 1 ligne = 1 jour (heure de Boston)

df_gold_2 = spark.sql("""
    WITH par_jour AS (
        SELECT date_course,
               COUNT(*)                          AS nb_courses,
               ROUND(SUM(price), 2)              AS ca_total,
               ROUND(AVG(price), 2)              AS prix_moyen,
               PERCENTILE_APPROX(price, 0.5)     AS prix_median,
               ROUND(100 * AVG(is_surge), 2)     AS pct_majorees,
               ROUND(100 * AVG(is_pluie), 2)     AS pct_sous_pluie,
               ROUND(AVG(temp_c), 1)             AS temp_moyenne_c
        FROM v_silver
        GROUP BY date_course
    )
    SELECT *,
           DATEDIFF(date_course, LAG(date_course, 1) OVER (ORDER BY date_course))           AS jours_depuis_precedent,
           LAG(nb_courses, 1) OVER (ORDER BY date_course)                                     AS nb_courses_precedent,
           ROUND(100 * (nb_courses - LAG(nb_courses, 1) OVER (ORDER BY date_course))
                 / LAG(nb_courses, 1) OVER (ORDER BY date_course), 2)                        AS evol_courses_pct,
           ROUND(prix_moyen - LAG(prix_moyen, 1) OVER (ORDER BY date_course), 2)              AS evol_prix_moyen,
           ROUND(AVG(prix_moyen) OVER (ORDER BY date_course ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS prix_moyen_glissant_3j,
           ROUND(AVG(pct_majorees) OVER (ORDER BY date_course ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS pct_majorees_glissant_3j
    FROM par_jour
    ORDER BY date_course
""")
assert df_gold_2 is not None, "TODO : Écrivez la requête SQL temporelle avec LAG() dans `df_gold_2`."

df_gold_2.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_TIME_PATH)
df_gold_2.show(20, truncate=False)

+-----------+----------+---------+----------+-----------+------------+--------------+--------------+----------------------+--------------------+----------------+---------------+----------------------+------------------------+
|date_course|nb_courses|ca_total |prix_moyen|prix_median|pct_majorees|pct_sous_pluie|temp_moyenne_c|jours_depuis_precedent|nb_courses_precedent|evol_courses_pct|evol_prix_moyen|prix_moyen_glissant_3j|pct_majorees_glissant_3j|
+-----------+----------+---------+----------+-----------+------------+--------------+--------------+----------------------+--------------------+----------------+---------------+----------------------+------------------------+
|2018-11-25 |467       |7889.0   |16.89     |15.0       |3.85        |0.0           |4.8           |NULL                  |NULL                |NULL            |NULL           |16.89                 |3.85                    |
|2018-11-26 |44071     |728469.5 |16.53     |13.5       |3.09        |56.45         |6.4        

**Lecture de la table Gold #2 (`/lakehouse/gold/evolution_journaliere`)**
- **Le prix est stable dans le temps :** le prix moyen journalier reste entre **16,27 $ et 16,89 $**, la moyenne glissante sur 3 jours entre 16,45 $ et 16,71 $, et le prix médian vaut 13,50 $ presque tous les jours. Pourtant, la part de courses sous la pluie varie de **0 % à 56 %** selon les jours : le prix ne suit pas la météo.
- **Piège des trous de collecte :** `jours_depuis_precedent` vaut **5** entre le 4 et le 9 décembre et **3** entre le 9 et le 12. Les variations calculées par `LAG()` sur ces lignes comparent des jours éloignés : ce ne sont pas des évolutions « d'un jour à l'autre ».
- **Piège des journées partielles :** le 25/11 (467 courses, la collecte commence à 22 h 40 heure de Boston), le 04/12, le 09/12, le 12/12 et le 18/12 sont incomplets. Les `evol_courses_pct` extrêmes (+9 337 %, −92 %) mesurent **la collecte, pas la demande**. Il faut donc analyser les prix (moyenne, médiane, % de majoration), pas les volumes.
- Les courses majorées (toutes chez Lyft) représentent entre **2,8 % et 3,9 %** de l'ensemble des courses chaque jour, sans lien visible avec les jours de pluie.

## 4. Table Gold #3 (Axe complémentaire libre) & Compaction `OPTIMIZE`

### Exercice 4.1 — Produire un 3e axe d'analyse (`ROW_NUMBER() OVER (PARTITION BY ...)`) et compacter Silver
1. Créez une 3e table Gold répondant à une question spécifique de votre problématique (ex. top sous-catégorie par région/groupe via `ROW_NUMBER() OVER (PARTITION BY groupe ORDER BY ... DESC) = 1`).
2. Exécutez `OPTIMIZE delta.'{SILVER_MAIN_PATH}'` et expliquez pourquoi la multiplication de micro-fichiers pénalise le NameNode HDFS.

In [5]:
# Gold #3 : pour chaque quartier de départ et chaque gamme, quelle compagnie est la moins chère au mile ?
GOLD_DUEL_PATH = "/lakehouse/gold/offre_moins_chere_par_quartier"
df_gold_3 = spark.sql("""
    WITH offre_quartier AS (
        SELECT source, gamme, cab_type, name,
               COUNT(*)                                 AS nb_courses,
               PERCENTILE_APPROX(prix_par_mile, 0.5)    AS prix_mile_median,
               PERCENTILE_APPROX(price, 0.5)            AS prix_median
        FROM v_silver
        GROUP BY source, gamme, cab_type, name
    ),
    classement AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY source, gamme ORDER BY prix_mile_median ASC, prix_median ASC) AS rn,
               COUNT(*)     OVER (PARTITION BY source, gamme)                                               AS nb_offres_en_lice
        FROM offre_quartier
    )
    SELECT source AS quartier_depart, gamme, cab_type AS compagnie_gagnante, name AS offre_gagnante,
           prix_mile_median, prix_median, nb_courses, nb_offres_en_lice
    FROM classement
    WHERE rn = 1
    ORDER BY quartier_depart, gamme
""")
df_gold_3.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_DUEL_PATH)
df_gold_3.show(12, truncate=False)
print("Victoires par compagnie (offres en concurrence uniquement) :")
df_gold_3.filter("nb_offres_en_lice > 1").groupBy("compagnie_gagnante").count().show()

# Tables complémentaires pour le dashboard (petites agrégations prêtes à l'emploi)
GOLD_HORAIRE_PATH = "/lakehouse/gold/profil_horaire"
spark.sql("""
    SELECT heure_boston, tranche_horaire, cab_type,
           COUNT(*) AS nb_courses, PERCENTILE_APPROX(price, 0.5) AS prix_median,
           PERCENTILE_APPROX(prix_par_mile, 0.5) AS prix_mile_median, ROUND(100 * AVG(is_surge), 2) AS pct_majorees
    FROM v_silver GROUP BY heure_boston, tranche_horaire, cab_type ORDER BY heure_boston, cab_type
""").coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_HORAIRE_PATH)

GOLD_METEO_PATH = "/lakehouse/gold/effet_meteo"
spark.sql("""
    SELECT cab_type, gamme, CASE WHEN is_pluie = 1 THEN 'Pluie' ELSE 'Sec' END AS meteo,
           COUNT(*) AS nb_courses, PERCENTILE_APPROX(price, 0.5) AS prix_median,
           PERCENTILE_APPROX(prix_par_mile, 0.5) AS prix_mile_median, ROUND(100 * AVG(is_surge), 2) AS pct_majorees
    FROM v_silver GROUP BY cab_type, gamme, is_pluie ORDER BY cab_type, gamme, meteo
""").coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_METEO_PATH)

# Compaction : nombre de fichiers de la table Silver avant / après OPTIMIZE
fichiers_avant = spark.sql(f"DESCRIBE DETAIL delta.`{SILVER_MAIN_PATH}`").first()["numFiles"]
print(f"Fichiers Parquet actifs de la table Silver AVANT OPTIMIZE : {fichiers_avant}")

spark.sql(f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`").show(truncate=False)
fichiers_apres = spark.sql(f"DESCRIBE DETAIL delta.`{SILVER_MAIN_PATH}`").first()["numFiles"]
print(f"Fichiers Parquet actifs APRÈS OPTIMIZE : {fichiers_apres}")

+---------------+-----------------------+------------------+--------------+----------------+-----------+----------+-----------------+
|quartier_depart|gamme                  |compagnie_gagnante|offre_gagnante|prix_mile_median|prix_median|nb_courses|nb_offres_en_lice|
+---------------+-----------------------+------------------+--------------+----------------+-----------+----------+-----------------+
|Back Bay       |Accessible PMR         |Uber              |WAV           |5.32            |9.5        |4591      |1                |
|Back Bay       |Grande capacité        |Uber              |UberXL        |8.51            |15.0       |4591      |2                |
|Back Bay       |Partagé                |Lyft              |Shared        |3.29            |5.0        |4275      |2                |
|Back Bay       |Premium                |Lyft              |Lux           |9.38            |16.5       |4276      |3                |
|Back Bay       |Premium grande capacité|Uber              |Bl

+------------------+-----+
|compagnie_gagnante|count|
+------------------+-----+
|              Lyft|   46|
|              Uber|   14|
+------------------+-----+



Fichiers Parquet actifs de la table Silver AVANT OPTIMIZE : 39


+-------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                       |metrics                                                                                                                                                                                           |
+-------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|hdfs://namenode:9000/lakehouse/silver/rides|{1, 39, {45537139, 45537139, 4.5537139E7, 1, 45537139}, {510696, 2616753, 1252395.8974358975, 39, 48843440}, 1, NULL, NULL, 1, 1, 39, 0, false, 0, 0, 1795001626223, 0, 22, 0, NULL, NULL, 42, 32}|
+-----------------------------------

Fichiers Parquet actifs APRÈS OPTIMIZE : 1


**Lecture de la table Gold #3 (`/lakehouse/gold/offre_moins_chere_par_quartier`)**
- Pour chaque quartier de départ et chaque gamme, `ROW_NUMBER() OVER (PARTITION BY quartier, gamme ORDER BY prix médian au mile)` garde l'offre la moins chère.
- Sur les **60 duels** (12 quartiers × 5 gammes) où les deux compagnies sont en concurrence, **Lyft gagne 46 fois, Uber 14 fois** :

| Gamme | Lyft moins cher | Uber moins cher |
|---|---|---|
| Partagé (Shared vs UberPool) | **12** | 0 |
| Premium (Lux, Lux Black vs Black) | **12** | 0 |
| Grande capacité (Lyft XL vs UberXL) | **10** | 2 |
| Standard (Lyft vs UberX) | **8** | 4 |
| Premium grande capacité (Lux Black XL vs Black SUV) | 4 | **8** |

  Seul le très haut de gamme (Black SUV) tourne à l'avantage d'Uber.
- L'offre « Accessible PMR » (WAV) n'existe que chez Uber (1 seule offre en lice).

**Tables complémentaires pour le dashboard :** `profil_horaire` (heure de Boston × compagnie) et `effet_meteo` (compagnie × gamme × pluie / sec), deux petites agrégations lues directement par Streamlit.

**Compaction `OPTIMIZE` :** la table Silver comptait **39 fichiers Parquet actifs** (22 écrits au Jour 2, plus ceux réécrits par le `MERGE`). Après `OPTIMIZE`, il n'en reste **qu'un seul** (45,5 Mo). Chaque fichier, et chacun de ses blocs, est un objet que le **NameNode garde en mémoire** (~150 octets par objet) et que Spark doit ouvrir (une tâche et une requête au NameNode par fichier). Des milliers de micro-fichiers saturent donc la mémoire du NameNode et ralentissent les lectures : c'est le *Small Files Problem*. Les anciens fichiers restent sur disque (Time Travel) jusqu'à un `VACUUM`.